# Opdracht 4 · Time series & forecasting
### Regressie-training · Antwoordmodel-versie

Bij tijdreeksen doet de **volgorde** van de data ertoe. In deze opdracht ga je:

1. Een tijdreeks **visualiseren** en trend + seizoen herkennen
2. **Lag-features** maken (waarden uit het verleden als input)
3. Een **time-based split** maken — bewust **géén** shuffle
4. Een regressiemodel trainen op de lag-features
5. Vergelijken met een **naïeve baseline** ("morgen = vandaag")

> **Dataset:** we genereren zelf een dagelijkse tijdreeks (~2 jaar) met een stijgende trend en een
> wekelijks patroon. Zo zijn de concepten goed zichtbaar en is geen download nodig.

> **Hoe werkt dit notebook?** De meeste cellen zijn ingevuld; op de plekken met **`# TODO`** vul jij iets in.


## 0. Voorbereiding & data genereren
Imports en het aanmaken van de tijdreeks. Gewoon uitvoeren — je hoeft hier niets aan te passen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

plt.rcParams["figure.figsize"] = (10, 4)
print("Bibliotheken geladen.")

In [ ]:
# Synthetische dagelijkse tijdreeks: trend + wekelijks seizoen + ruis
rng = np.random.default_rng(42)
n = 730  # ongeveer twee jaar aan dagen

t = np.arange(n)
trend = 0.05 * t                      # langzaam stijgende trend
seizoen = 8 * np.sin(2 * np.pi * t / 7)  # wekelijks patroon (periode 7 dagen)
ruis = rng.normal(0, 2, n)            # willekeurige ruis
waarde = 50 + trend + seizoen + ruis

datum = pd.date_range("2023-01-01", periods=n, freq="D")
df = pd.DataFrame({"datum": datum, "waarde": waarde})
df.head()

## 1. De tijdreeks visualiseren
Voordat je modelleert, kijk je altijd eerst naar de data. De plot is al voor je gemaakt — voer de cel uit.

In [ ]:
# Plot van de hele reeks (al ingevuld)
plt.plot(df["datum"], df["waarde"], color="#4473C5", linewidth=1)
plt.title("De volledige tijdreeks")
plt.xlabel("Datum"); plt.ylabel("Waarde")
plt.show()

De stijgende lijn is de **trend**. Het golvende patroon is het **seizoen**. Om het seizoen beter te zien,
zoomen we in op de eerste 8 weken.

In [ ]:
# Inzoomen op de eerste 56 dagen om het wekelijkse patroon te zien (al ingevuld)
plt.plot(df["datum"][:56], df["waarde"][:56], marker="o", color="#4473C5")
plt.title("Eerste 8 weken - let op het wekelijkse patroon")
plt.xlabel("Datum"); plt.ylabel("Waarde")
plt.show()

> **Kijk goed:** zie je de stijgende trend in de eerste plot, en het zich herhalende wekelijkse
> patroon in de tweede? Dat patroon gaan we straks benutten met een lag van 7 dagen.

## 2. Lag-features maken
Een **lag-feature** is simpelweg een eerdere waarde van de reeks, die we als input gebruiken.
- `lag1` = de waarde van **gisteren**
- `lag7` = de waarde van **vorige week, dezelfde dag** (vangt het wekelijkse patroon)

### TODO 1 — Maak de lag-features
Maak twee nieuwe kolommen met `.shift(...)`:
- `lag1` = `waarde` één dag verschoven
- `lag7` = `waarde` zeven dagen verschoven

> Tip: `df["waarde"].shift(1)` geeft de waarde van de vorige rij.

In [ ]:
# ANTWOORD
df["lag1"] = df["waarde"].shift(1)
df["lag7"] = df["waarde"].shift(7)

# De eerste rijen hebben nu lege lag-waarden; die verwijderen we.
df = df.dropna().reset_index(drop=True)
df.head()

## 3. Time-based split
Bij tijdreeksen mag je **niet** willekeurig splitsen: dan zou je toekomst gebruiken om het verleden te
voorspellen (data-lekkage). In plaats daarvan trainen we op het **eerste deel** en testen op het
**laatste deel** van de reeks.

### TODO 2 — Maak de time-based split
Bepaal de splitsgrens op 80% van de rijen, en gebruik de **eerste** 80% als train en de **laatste** 20% als test.
**Niet** shuffelen!

> Tip: `split = int(len(df) * 0.8)`, dan `df.iloc[:split]` voor train en `df.iloc[split:]` voor test.

In [ ]:
# ANTWOORD
split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

print("Train:", len(train), "dagen (van", train['datum'].min().date(), "tot", train['datum'].max().date(), ")")
print("Test :", len(test), "dagen (van", test['datum'].min().date(), "tot", test['datum'].max().date(), ")")

## 4. Een forecast-model trainen
We voorspellen de waarde van vandaag op basis van `lag1` en `lag7`.

### TODO 3 — Train het model
Train een `LinearRegression`-model op de **lag-features** van de train-set, en voorspel de test-set.
De featurelijst is al voor je gedefinieerd.

> Tip: `model.fit(train[features], train["waarde"])` en `model.predict(test[features])`.

In [ ]:
# ANTWOORD
features = ["lag1", "lag7"]

model = LinearRegression()
model.fit(train[features], train["waarde"])
voorspelling = model.predict(test[features])

mae_model = mean_absolute_error(test["waarde"], voorspelling)
print(f"MAE van het model: {mae_model:.3f}")

## 5. Vergelijken met een naïeve baseline
De simpelste forecast is: **"morgen is gelijk aan vandaag"**. Dat is precies de `lag1`-kolom.
Als ons model deze baseline niet verslaat, voegt het niets toe.

### TODO 4 — Bereken de baseline-MAE
De naïeve voorspelling voor de test-set is de kolom `lag1` (de waarde van de vorige dag).
Bereken de MAE tussen `test["waarde"]` en `test["lag1"]`.

> Tip: `mean_absolute_error(test["waarde"], test["lag1"])`.

In [ ]:
# ANTWOORD
mae_naive = mean_absolute_error(test["waarde"], test["lag1"])
print(f"MAE van het model   : {mae_model:.3f}")
print(f"MAE van de baseline : {mae_naive:.3f}")
print("Model verslaat de baseline!" if mae_model < mae_naive else "Baseline is beter...")

Tot slot zetten we de voorspelling naast de werkelijke waarden (al ingevuld).

In [ ]:
# Voorspelling vs. werkelijkheid op de test-periode (al ingevuld)
plt.plot(test["datum"], test["waarde"], color="#4473C5", label="werkelijk", linewidth=1.5)
plt.plot(test["datum"], voorspelling, color="#D97733", label="voorspeld", linewidth=1.5)
plt.title("Test-periode: voorspelling vs. werkelijkheid")
plt.xlabel("Datum"); plt.ylabel("Waarde")
plt.legend()
plt.show()

## 6. Bespreking
Bespreek met je buur:

- Verslaat het model de naïeve baseline? Met hoeveel?
- Welke lag-feature is belangrijker, denk je — `lag1` of `lag7`? Hoe zie je dat terug in de data?
- Waarom mag je bij tijdreeksen de data niet willekeurig shuffelen?
- Wat zou er gebeuren met je MAE als je de seizoen-lag (`lag7`) zou weglaten?

---
### Toelichting voor de trainer
- Verwacht **MAE model ≈ 2,3** vs. **MAE baseline ≈ 4,8**: het model verslaat de naïeve baseline ruim.
- De coëfficiënt van **`lag7` is veel groter dan die van `lag1`** (≈ 0,86 vs. 0,12), omdat de reeks een
  sterk wekelijks seizoen heeft — "dezelfde dag vorige week" is daardoor het meest voorspellend.
- Kernpunt **data-lekkage**: bij een willekeurige shuffle zou het model toekomstige punten "zien" en
  onrealistisch goed scoren. De time-based split benadert het gebruik in productie.
- Optioneel uitbreiden: laat cursisten `lag7` weglaten en zien dat de MAE stijgt richting de baseline.